In [ ]:
import sys
import os
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))

In [ ]:
from utils.dataloader import get_dataloader, get_transforms

BASE_DIR = os.path.abspath("..")

train_path = os.path.join(BASE_DIR, "data/WLBisindo/split/train")
val_path   = os.path.join(BASE_DIR, "data/WLBisindo/split/val")
test_path  = os.path.join(BASE_DIR, "data/WLBisindo/split/test")

train_transform, val_transform = get_transforms()

train_loader = get_dataloader(train_path, transform=train_transform)
val_loader   = get_dataloader(val_path, shuffle=False, transform=val_transform)
test_loader  = get_dataloader(test_path, shuffle=False, transform=val_transform)

In [ ]:
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

log_dir = os.path.join(BASE_DIR, "outputs/logs/mobile_net", f"run_{timestamp}")
os.makedirs(log_dir, exist_ok=True)

writer = SummaryWriter(log_dir)

In [ ]:
log_file = os.path.join(log_dir, "log.txt")

def log_message(msg):
    print(msg)
    with open(log_file, "a") as f:
        f.write(msg + "\n")

In [ ]:
from models.mobile_net import MobileNetTransformer

NUM_CLASSES = len(train_loader.dataset.label_map)

model = MobileNetTransformer(num_classes=NUM_CLASSES)

In [ ]:
import torch
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# freeze CNN
for param in model.encoder.features.parameters():
    param.requires_grad = False

# unfreeze last block
for param in model.encoder.features[-1].parameters():
    param.requires_grad = True

In [ ]:
labels = train_loader.dataset.labels

class_counts = Counter(labels)
total = sum(class_counts.values())

weights = [total / class_counts[i] for i in range(len(class_counts))]
weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=weights)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-5,   # lebih kecil dari CNN-LSTM
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2,
    factor=0.5
)

In [ ]:
EPOCHS = 10
patience = 3
counter = 0
best_val_loss = float("inf")

import json

config = {
    "model": "MobileNetTransformer",
    "epochs": 10,
    "lr": 3e-5,
    "weight_decay": 1e-5,
    "batch_size": train_loader.batch_size,
    "num_classes": NUM_CLASSES,
    "transformer_layers": 2
}

with open(os.path.join(log_dir, "config.json"), "w") as f:
    json.dump(config, f, indent=4)

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ===== TRAIN =====
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc="Training", leave=False)

    for x, y in train_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

        # update progress bar
        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total


    # ===== VALIDATION =====
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    val_bar = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():
        for x, y in val_bar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            outputs = model(x)
            loss = criterion(outputs, y)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

            # update progress bar
            val_bar.set_postfix({
                "loss": f"{loss.item():.4f}"
            })

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # ===== LOGGING =====
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)

    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)

    writer.add_scalar("Overfitting/gap", train_acc - val_acc, epoch)

    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar("LR", current_lr, epoch)

    for name, param in model.named_parameters():
        writer.add_histogram(f"Weights/{name}", param, epoch)

        if param.grad is not None:
            writer.add_histogram(f"Gradients/{name}", param.grad, epoch)

    log_message(f"Epoch {epoch+1}/{EPOCHS}")
    log_message(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    log_message(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    log_message(f"LR: {current_lr}")
    log_message("-" * 40)

    # ===== SCHEDULER =====
    scheduler.step(val_loss)


    # ===== EARLY STOPPING =====
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), os.path.join(log_dir, "best_model.pth"))
        log_message("Best model saved!")

    else:
        counter += 1
        log_message(f"EarlyStopping counter: {counter}/{patience}")

        if counter >= patience:
            log_message("Early stopping triggered!")
            break


# SAVE LAST MODEL
torch.save(model.state_dict(), os.path.join(log_dir, "last_model.pth"))

In [ ]:
from utils.evaluate import evaluate

report_path = os.path.join(log_dir, "evaluation.json")
metrics = evaluate(model, test_loader, device, save_path=report_path)